In [25]:
import requests
import re
import json
from bs4 import BeautifulSoup
from urllib.parse import quote
from tqdm import tqdm

In [26]:
dict_steam_games = {}

header = {
    'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Whale/4.29.282.14 Safari/537.36'
    }

# 리뷰의 평가 기준 4개
reviews_tags = ['funny','summary','all','recent']

# 오픈월드, dlc 포함 x, 상위 100개를 불러옴
URL = 'https://store.steampowered.com/search/results/?query&start=0&count=100&dynamic_data=&sort_by=_ASC&supportedlang=koreana&tags=1695&category1=998&snr=1_7_7_7000_7&filter=topsellers&infinite=1'

def games_info(link, result):
    '''2024년 이하의 게임 고유 번호, 타이틀명, 원가, 주소를 반환'''
    response = requests.get(link, headers=header)
    json_data = response.json()
    soup = BeautifulSoup(json_data['results_html'], 'html.parser')

    # 게임 고유 번호, 타이틀명, 원가, 주소 저장
    price = soup.select('a.search_result_row')
    game_cnt = 0

    for game in price:
        if int(game.select_one('div.search_released').text.strip()[-4:]) >= 2025:
            continue

        original_price = game.select_one('div.search_discount_and_price .discount_original_price')

        if original_price:
            # 원래 가격이 있을 경우
            price_value = original_price.text.replace('₩','').replace(',','').strip() 
            game_title =re.sub(r'[^\w\s_]','',game.select_one('span.title').text.replace(' ','_')) # 게임 타이틀을 저장
            game_code = game.get('data-ds-itemkey')[4:] # 게임 고유 번호를 저장
            game_link = game.get('href')
        else:
            # 원래 가격이 없으면 할인된 가격을 사용
            final_price = game.select_one(' div.search_discount_and_price .discount_final_price')
            price_value = final_price.text.replace('₩','').replace(',','').strip()
            game_title =re.sub(r'[^\w\s_]','',game.select_one('span.title').text.replace(' ','_'))
            game_code = game.get('data-ds-itemkey')[4:]
            game_link = game.get('href')

            if price_value == 'Free':
                price_value = 0

        result[game_title] = {}
        sub_result = result[game_title]
        sub_result['code'] = game_code
        sub_result['price'] = int(price_value)
        sub_result['link'] = game_link
    
        game_cnt += 1
        if game_cnt == 50:
            break

    return result

def games_time_avg(code):
    '''재밌는, 유용한, 최근의, 모든 평가 각각의 기준별로 상위 100개의 평가를 한 플레이어의 시간 합, 평균을 구함'''
    total_time = []

    for tag in reviews_tags:
        cursor = '*'
        for _ in range(5):
            URL = f'https://store.steampowered.com/appreviews/{code}?use_review_quality=1&cursor={cursor}&day_range=30&start_date=-1&end_date=-1&date_range_type=all&filter={tag}&language=english&l=english&review_type=all&purchase_type=all&playtime_filter_min=0&playtime_filter_max=0&playtime_type=all&filter_offtopic_activity=1&summary_num_positive_reviews=292163&summary_num_reviews=310508'
            response = requests.get(URL, headers=header)
            reviews = response.json()
            cursor = quote(reviews['cursor'])
            soup = BeautifulSoup(reviews['html'],'html.parser')
        
            total_time += [float(time.text.strip().split(' ')[0].replace(',','')) for time in soup.select('div.hours')]
    try:
        time_avg = round(sum(total_time) / len(total_time), 2)
    # 평가를 하지 못하는 확장판, DLC 등의 에러 처리
    except ZeroDivisionError:
        time_avg = 'Error'
    return time_avg

# 게임의 기본적인 정보 크롤링
dict_steam_games = games_info(URL, dict_steam_games)      

In [27]:
# 해당 게임의 평가 기준 총 플레이 시간의 평균을 반환
for key, value in tqdm(dict_steam_games.items(), desc="Processing Games"):
    value['time_avg'] = games_time_avg(value['code'])

Processing Games:   0%|          | 0/50 [00:00<?, ?it/s]

Processing Games: 100%|██████████| 50/50 [08:47<00:00, 10.56s/it]


In [28]:
# 평가가 없는 (플레이 시간이 에러인) 게임 제외
dict_filtered = {k: v for k, v in dict_steam_games.items() if v['time_avg']!='Error'}
dict_filtered

{'Delta_Force': {'code': '2507950',
  'price': 0,
  'link': 'https://store.steampowered.com/app/2507950/Delta_Force/?snr=1_7_7_7000_150_1',
  'time_avg': 78.32},
 'Palworld': {'code': '1623730',
  'price': 32000,
  'link': 'https://store.steampowered.com/app/1623730/Palworld/?snr=1_7_7_7000_150_1',
  'time_avg': 138.05},
 'Red_Dead_Redemption_2': {'code': '1174180',
  'price': 73000,
  'link': 'https://store.steampowered.com/app/1174180/Red_Dead_Redemption_2/?snr=1_7_7_7000_150_1',
  'time_avg': 137.01},
 'Grand_Theft_Auto_V': {'code': '271590',
  'price': 26400,
  'link': 'https://store.steampowered.com/app/271590/Grand_Theft_Auto_V/?snr=1_7_7_7000_150_1',
  'time_avg': 311.64},
 'Once_Human': {'code': '2139460',
  'price': 0,
  'link': 'https://store.steampowered.com/app/2139460/Once_Human/?snr=1_7_7_7000_150_1',
  'time_avg': 134.3},
 'Cyberpunk_2077': {'code': '1091500',
  'price': 66000,
  'link': 'https://store.steampowered.com/app/1091500/Cyberpunk_2077/?snr=1_7_7_7000_150_1',
 

In [45]:
def achievement_info(value):
    url = f'https://steamcommunity.com/stats/{value['code']}/achievements'
    response = requests.get(url, headers=header)
    soup = BeautifulSoup(response.text, 'html.parser')
    try:
        total_ach = int(soup.select_one('div.maincontent span').text) # 총 업적 개수

        value['total_ach'] = total_ach
        value['ach_list'] = {}
        ach_list = value['ach_list']
        
        for ach in soup.select('div.maincontent div.achieveRow'):
            ach_title = ach.select_one('div.achieveTxt').text.strip().split('\n')[0] # 0번 업적 이름, 1번 업적 내용
            ach_percent = ach.select_one('div.achievePercent').text # 해당 업적 달성률
            ach_list[ach_title] = ach_percent
    # 업적이 없는 게임일 경우
    except:
        value['total_ach'] = 0

    return value

In [46]:
for k,v in tqdm(dict_filtered.items(), desc="Processing"):
    dict_filtered[k] = achievement_info(v)

Processing: 100%|██████████| 47/47 [00:17<00:00,  2.76it/s]


In [47]:
dict_filtered

{'Delta_Force': {'code': '2507950',
  'price': 0,
  'link': 'https://store.steampowered.com/app/2507950/Delta_Force/?snr=1_7_7_7000_150_1',
  'time_avg': 78.32,
  'total_ach': 53,
  'ach_list': {'First Victory': '17.6%',
   'Field Vanguard': '10.4%',
   'Field Half-Marathon': '10.1%',
   'Field Marathon': '7.3%',
   'Veteran Operator': '3.6%',
   'Tank Terminator': '3.6%',
   'Endless Barrage': '3.4%',
   'Field Marshal ': '3.3%',
   'Ace Sniper': '2.5%',
   'Brickyard Supervisor': '1.9%',
   'Nuclear Strike': '1.6%',
   'My Turf': '1.1%',
   'Welding Warlord': '1.0%',
   'Top Agent': '1.0%',
   'Ghost in the Halls': '0.9%',
   'Headshot Expert': '0.7%',
   'Killing Machine': '0.4%',
   'Multi-Core Processing': '0.3%',
   'Military Enthusiast': '0.2%',
   'Precise Detection': '0.2%',
   'Next-Gen Material': '0.2%',
   'Rescue More!': '0.1%',
   'Hackclaw - Operations': '0.1%',
   'Everlasting Heart': '0.1%',
   'Luna - Warfare': '0.1%',
   'Vyron - Operations': '0.1%',
   'D-wolf - War

In [48]:
# json 형식으로 저장
with open("steam_game_dict.json", "w", encoding="utf-8") as json_file:
    json.dump(dict_filtered, json_file, ensure_ascii=False, indent=4)

In [ ]:
# json 파일 불러오기
# with open("steam_game_dict.json", "r", encoding="utf-8") as json_file:
#      dict_filtered = json.load(json_file)